In [1]:
# Run this in a cell before exporting
%pip install onnx onnxsim onnxruntime-gpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from ultralytics import YOLO
import yaml
import torch
torch.cuda.empty_cache()


DATASET_PATH = "augmentation_dataset"

In [3]:
def create_dataset_yaml(dataset_path="augmentation_dataset", num_classes=1, class_names=None):
    """Create dataset YAML configuration with proper formatting"""
    if class_names is None:
        class_names = {0: 'pest'}
    
    yaml_content = {
        'path': os.path.abspath(dataset_path),
        'train': 'train/images',
        'val': 'valid/images', 
        'test': 'test/images',
        'nc': num_classes,
        'names': class_names
    }
    
    # Save YAML
    with open("data.yaml", "w") as f:
        yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)
    
    print("✅ data.yaml created!")
    print(f"Dataset path: {os.path.abspath(dataset_path)}")
    return "data.yaml"

# Create dataset configuration
yaml_path = create_dataset_yaml(DATASET_PATH, num_classes=2, class_names={0: 'raccoon', 1: 'rodent'}) # changed to try two-class system for better precision

✅ data.yaml created!
Dataset path: c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset


In [4]:
def verify_dataset_structure(dataset_path):
    """Verify dataset has correct structure"""
    required_dirs = [
        "train/images", "train/labels",
        "valid/images", "valid/labels", 
        "test/images", "test/labels"
    ]
    
    print("🔍 Verifying dataset structure...")
    for dir_path in required_dirs:
        full_path = os.path.join(dataset_path, dir_path)
        if not os.path.exists(full_path):
            print(f"❌ Missing: {full_path}")
            return False
        # Check if directory has files
        files = os.listdir(full_path)
        if not files:
            print(f"⚠️  Empty directory: {full_path}")
        else:
            print(f"✅ {dir_path}: {len(files)} files")
    
    return True

# Verify dataset
dataset_valid = verify_dataset_structure(DATASET_PATH)
if not dataset_valid:
    print("❌ Dataset structure invalid! Fix before training.")

🔍 Verifying dataset structure...
✅ train/images: 4966 files
✅ train/labels: 4966 files
✅ valid/images: 356 files
✅ valid/labels: 356 files
✅ test/images: 178 files
✅ test/labels: 178 files


In [5]:
def train_model(yaml_path="data.yaml"):
    """Main training function with enhanced configuration"""

    # Load YOLO11 model
    print("🚀 Loading YOLO11n model...")
    model = YOLO("yolo11n.pt")
    
    # Training configuration
    results = model.train(
        data=yaml_path,
        epochs=50,
        imgsz=640,
        batch=32,
        device=0,
        single_cls=True,       # Train as single-class dataset
        verbose=True,          # Print results per epoch
    )
    
    return model, results

# Train the model (this will take ~25 minutes)
model, results = train_model(yaml_path)

🚀 Loading YOLO11n model...
New https://pypi.org/project/ultralytics/8.3.224 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=

In [6]:
def simple_evaluate_model(model, yaml_path="data.yaml"):
    """Simple baseline evaluation"""
    print("📊 Evaluating model...")
    metrics = model.val(data=yaml_path)

    return metrics


# Run validation
val_metrics = simple_evaluate_model(model)


📊 Evaluating model...
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1008.2622.5 MB/s, size: 66.1 KB)
val: Scanning C:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\valid\labels.cache... 356 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 356/356 355.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 5.5it/s 2.2s0.1s
                   all        356        365      0.994      0.995      0.994      0.887
Speed: 0.4ms preprocess, 2.5ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to C:\Users\plebj\Desktop\School\CS5814\runs\detect\train2


In [7]:
def evaluate_model(model, yaml_path="data.yaml"):
    """Comprehensive model evaluation"""
    print("📊 Evaluating model...")
    
    # Validation metrics
    metrics = model.val(
        data=yaml_path,
        split="val",
        save_json=True,    # Save results to JSON
        save_hybrid=True,  # Save hybrid version of labels
        conf=0.5,          # Object confidence threshold
        iou=0.6,           # NMS IoU threshold
        plots=True         # Save plots
    )
    
    # Test set evaluation  
    test_metrics = model.val(
        data=yaml_path,
        split="test",
        conf=0.5,
        iou=0.6
    )
    
    return metrics, test_metrics

# Evaluate the trained model
val_metrics, test_metrics = evaluate_model(model)

📊 Evaluating model...
WARNING 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 1069.1547.1 MB/s, size: 49.6 KB)
val: Scanning C:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\valid\labels.cache... 356 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 356/356  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 6.1it/s 2.0s0.1s
                   all        356        365      0.994      0.995      0.994      0.894
Speed: 0.3ms preprocess, 2.1ms inference, 0.0ms loss, 0.7ms postprocess per image
Saving C:\Users\plebj\Desktop\School\CS5814\runs\detect\train3\predictions.json...
Results saved to C:\Users\plebj\Desktop\School\CS5814\runs\detect\train3
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
val: Fast im

In [8]:
def export_model(model, export_formats=["onnx", "tflite"]):
    """Export model to various formats"""
    print("📤 Exporting model...")
    
    exported_paths = {}
    
    for format in export_formats:
        try:
            if format == "tflite":
                # TFLite export for Raspberry Pi
                path = model.export(format=format, int8=True, data="data.yaml")
            else:
                path = model.export(format=format)
            
            exported_paths[format] = path
            print(f"✅ Exported to {format}: {path}")
            
        except Exception as e:
            print(f"❌ Failed to export {format}: {e}")
    
    return exported_paths

# Export the model
exported_paths = export_model(model, export_formats=["onnx"])  # "tflite" if needed

📤 Exporting model...
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CPU (AMD Ryzen 7 9700X 8-Core Processor)

PyTorch: starting from 'C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)

ONNX: starting export with onnx 1.19.0 opset 19...
ONNX: slimming with onnxslim 0.1.70...
ONNX: export success  1.2s, saved as 'C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.onnx' (10.1 MB)

Export complete (1.3s)
Results saved to C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights
Predict:         yolo predict task=detect model=C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.onnx imgsz=640 data=data.yaml  
Visualize:       https://netron.app
✅ Exported to onnx: C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\we

In [9]:
def test_inference(model, dataset_path="augmentation_dataset"):
    """Test model inference on sample images"""
    print("🧪 Testing inference...")
    
    test_dir = os.path.join(dataset_path, "test/images")
    if os.path.exists(test_dir):
        test_images = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        if test_images:
            # Test on first 3 images
            for i, img_name in enumerate(test_images[:3]):
                img_path = os.path.join(test_dir, img_name)
                print(f"Testing: {img_name}")
                
                results = model(img_path, conf=0.60)  # Confidence threshold
                
                # Show results (optional - comment out for headless environments)
                # results[0].show()
                
                # Print detection info
                for r in results:
                    if len(r.boxes) > 0:
                        print(f"  Detected {len(r.boxes)} pests")
                    else:
                        print("  No pests detected")
            
            print("✅ Inference test completed!")
        else:
            print("⚠️  No test images found")
    else:
        print("⚠️  Test directory not found")

# Test inference
test_inference(model, DATASET_PATH)

🧪 Testing inference...
Testing: 00b8f682-b70d-4dc8-a813-e6c2e8baa232.jpg

image 1/1 c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\images\00b8f682-b70d-4dc8-a813-e6c2e8baa232.jpg: 640x640 1 item, 5.1ms
Speed: 3.4ms preprocess, 5.1ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
  Detected 1 pests
Testing: 00cc44cb-d257-4ab4-a082-b77e5625b42a.jpg

image 1/1 c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\images\00cc44cb-d257-4ab4-a082-b77e5625b42a.jpg: 640x640 1 item, 5.1ms
Speed: 1.2ms preprocess, 5.1ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
  Detected 1 pests
Testing: 03012f1f-9ae3-434e-aa56-5043401b186b.jpg

image 1/1 c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\images\03012f1f-9ae3-434e-aa56-5043401b186b.jpg: 640x640 1 item, 4.9ms
Speed: 1.9ms preprocess, 4.9ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)
  Detected 1 pests
✅ Inference test completed!


In [10]:
print("\n🎉 Training completed successfully!")
print(f"📁 Model saved in: {os.path.join('runs', 'detect', 'train')}")
import json

# --- Print final metrics ---
if hasattr(results, 'results_dict'):
    print(f"📊 Final mAP@0.5: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")

if test_metrics is not None:
    # Prefer results_dict for broader version compatibility
    if hasattr(test_metrics, 'results_dict'):
        precision = test_metrics.results_dict.get('metrics/precision(B)', None)
        recall = test_metrics.results_dict.get('metrics/recall(B)', None)
        map50 = test_metrics.results_dict.get('metrics/mAP50(B)', None)
        map5095 = test_metrics.results_dict.get('metrics/mAP50-95(B)', None)

        print(f"📊 Test Precision: {precision:.4f}")
        print(f"📊 Test Recall: {recall:.4f}")
        print(f"📊 Test mAP@0.5: {map50:.4f}")
        print(f"📊 Test mAP@0.5:0.95: {map5095:.4f}")
    else:
        print("⚠️ test_metrics does not contain results_dict.")
else:
    print("📊 Test metrics not available (test_metrics is None)")





🎉 Training completed successfully!
📁 Model saved in: runs\detect\train
📊 Final mAP@0.5: 0.9943495954498118
📊 Test Precision: 0.9945
📊 Test Recall: 0.9940
📊 Test mAP@0.5: 0.9947
📊 Test mAP@0.5:0.95: 0.8931


In [11]:
# ============================================================
# Enhanced Evaluation Suite (Original + Augmented + Synthetic)
# ============================================================

import os
from pathlib import Path
from ultralytics import YOLO
import numpy as np
import matplotlib.pyplot as plt

# --- Config ---
MODEL_PATH = Path("runs/detect/train/weights/best.pt")
BASE_DATASET = Path("augmentation_dataset")  # dataset used during training

# --- Load model ---
print(f"📦 Loading YOLO model from {MODEL_PATH} ...")
model = YOLO(MODEL_PATH)
print("✅ Model loaded successfully!")

# ============================================================
# Test A: Evaluate on Original Test Split
# ============================================================
print("\n🧪 [Test A] Evaluating on Original Test Split...")
test_dir = BASE_DATASET / "test"
results_orig = model.val(
    data="data.yaml",
    split="test",
    conf=0.6,
    save_json=True,
    plots=True
)
print("✅ Original test split evaluation complete!")

# ============================================================
# Test B: Evaluate on Dynamically Augmented Test Images
# ============================================================
print("\n🧪 [Test B] Evaluating on Augmented Test Images (On-the-Fly)...")

import albumentations as A
import cv2
from tqdm import tqdm

aug = A.Compose([
    A.RandomBrightnessContrast(p=0.4),
    A.MotionBlur(blur_limit=5, p=0.3),
    A.GaussNoise(var_limit=(10, 40), p=0.3),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
])

aug_dir = Path("temp_aug_test")
aug_dir.mkdir(parents=True, exist_ok=True)

# Apply light augmentations to test images
for img_path in tqdm(list((BASE_DATASET / "test/images").glob("*.jpg")), desc="Augmenting test set"):
    img = cv2.imread(str(img_path))
    aug_img = aug(image=img)["image"]
    cv2.imwrite(str(aug_dir / img_path.name), aug_img)

results_aug = model.predict(
    source=str(aug_dir),
    conf=0.25,
    save=True
)
print("✅ Augmented test images evaluation complete!")

# ============================================================
# Combined Summary
# ============================================================
print("\n📊 Summary of Evaluation Results:")
try:
    print(f"Original Test: mAP@50 = {results_orig.box.map50:.3f}, Precision = {results_orig.box.mp:.3f}, Recall = {results_orig.box.mr:.3f}")
except:
    pass

print(f"Augmented Test Predictions Saved in: {results_aug[0].save_dir}")

print("\n💡 Tip: Compare metrics between A and B to assess generalization under noise/augmentation.")


📦 Loading YOLO model from runs\detect\train\weights\best.pt ...
✅ Model loaded successfully!

🧪 [Test A] Evaluating on Original Test Split...
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1089.7514.0 MB/s, size: 67.2 KB)
val: Scanning C:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\labels.cache... 178 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 178/178  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 10.8it/s 1.1s0.1s
                   all        178        181      0.994      0.994      0.995      0.893
Speed: 0.5ms preprocess, 2.5ms inference, 0.0ms loss, 0.5ms postprocess per image
Saving C:\Users\plebj\Desktop\School\CS5814\runs\detect\val\predictions.json...
Results saved to C:\Users\plebj\Desktop\School\CS581

Augmenting test set: 100%|██████████| 178/178 [00:00<00:00, 254.16it/s]


image 1/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\00b8f682-b70d-4dc8-a813-e6c2e8baa232.jpg: 640x640 1 item, 4.2ms
image 2/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\00cc44cb-d257-4ab4-a082-b77e5625b42a.jpg: 640x640 1 item, 3.7ms
image 3/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\03012f1f-9ae3-434e-aa56-5043401b186b.jpg: 640x640 1 item, 5.2ms
image 4/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\0658db41-7268-42ad-95f4-602055009777.jpg: 640x544 1 item, 33.6ms
image 5/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\0af45afe-65c2-448d-9e10-83c014f8a447.jpg: 640x640 1 item, 4.4ms


image 6/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\0c12ead6-a8df-422a-aaa8-306849202d98.jpg: 640x640 1 item, 3.8ms
image 7/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\0c771d62-e9a3-427f-abbb-595e22ccbebd.jpg: 640x640 1 item, 4.1ms
image 8/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\0f2499fe-9cf3-4679-9167-e45b2cc984ff.jpg: 640x640 1 item, 4.7ms
image 9/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\108249d8-37ce-4ba7-954f-d4664bdac346.jpg: 640x640 1 item, 8.9ms
image 10/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\111e2557-9c48-4bee-b588-05eade0c0ee5.jpg: 640x640 1 item, 4.0ms
image 11/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\12e20d5e-85e5-4da5-b999-b7f1590d8ba2.jpg: 640x640 1 item, 4.0ms
image 12/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\12ed456b-1dd9-49f1-8d2f-c659112c02cc.jpg: 640x640 1 item, 4.4ms
image 13/178 c:\Users\plebj\Desktop\School\CS5814\temp_aug_test\1484f44c-1839-49bd-947c-3023aaebab2e.jpg: 44